Detta är koden till chatbotten som används i inlämningsuppgiften

## Importer

In [154]:
from pypdf import PdfReader
from pathlib import Path
import re
from google import genai
import os
from google.genai import types
from dotenv import load_dotenv
import json
import time
import numpy as np


In [155]:
load_dotenv()

True

In [156]:
client = genai.Client(api_key=os.getenv("API_KEY"))

# RAG-delen

## Inläsning av dokumenten

In [157]:
# Läser in alla pdferna som ligger i mappen "Dokument till RAG" i mitt repository
folder_path = Path("Dokumenten till RAG")
pdf_files = list(folder_path.glob("*.pdf"))

documents = []

for pdf in folder_path.glob("*.pdf"): #läser in alla alla dokument som slutar med pdf
    reader = PdfReader(pdf)

    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        # Lägger bara till om det verkligen finns någon text på sidan
        if page_text:
            text += page_text + "\n"
    
    # sparar från vilket dokument texten kommer ifrån
    documents.append({
        "source": pdf.name,
        "text": text
    })


Vid steget kontroll av embeddingsen så upptäckte jag att textkvaliten var dålig då det var radbrytningar mitt i ord i pdfen så därför lägger jag till detta steget som ska städa texten för att underlätta embeddningsen i senare skede. 

In [158]:
def clean_text(text):
    text = text.replace("-\n", "")              # fixa vat-\nten → vatten
    text = text.replace("\n", " ")              # ta bort radbrytningar
    text = text.replace("–", "-")               # gör om kosntiga -
    text = text.replace("—", "-")               # gör om kosntiga -
    text = re.sub(r"\s+", " ", text)            #städar extra whitespace
    return text.strip()  

In [159]:
for doc in documents:
    doc["text"] = clean_text(doc["text"])

## Första chunkningen

In [160]:
chunks = []
n = 5
overlap = 3

for doc in documents: 
    sentences = re.split(r'(?<=[.!?]) +', doc["text"])

    for i in range(0, len(sentences), n - overlap):
        chunk_sentences = sentences[i:i+n]

        chunk_text = " ".join(chunk_sentences)

        chunks.append({
            "text": chunk_text,
            "source": doc["source"]
        })

In [161]:
print(len(chunks))

477


## Embeddings

In [162]:
def create_embeddings(text,
                      model="gemini-embedding-001",
                      task_type="SEMANTIC_SIMILARITY"):
    return client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )

Eftersom jag fick problem med att jag överskred min quota så har jag minskat antal manualer från 5 till 1 samt satt in att spara embeddnings mellan varje lyckad embedding för att inte behöva köra om från början om jag uppnår min quota. Jag satt även in en liten fördröjning mellan chunksen för att minska belastningen

In [163]:
save_path = "embeddings.json"

# ladda in redan sparade embeddings om filen finns
if os.path.exists(save_path):
    with open(save_path, "r", encoding="utf-8") as f:
        embeddings = json.load(f)    
else:
    embeddings = []

already_done = {item["text"] for item in embeddings}

for chunk in chunks:
    if chunk["text"] in already_done:
        continue

    try:
        response = create_embeddings(chunk["text"])
    
        item = {
        "text": chunk["text"],
        "source": chunk["source"],
        "embedding": response.embeddings[0].values
        }

        embeddings.append(item)

        # spara direkt efter varje lyckad embedding
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(embeddings, f, ensure_ascii=False, indent=2)

        print(f"Sparade chunk från {chunk['source']}")

        time.sleep(1) # liten paus för att minska belastningen

    except Exception as e:
        print("Stopp:", e)
        break

    # Gör om embeddings till numpy‑matris (för beräkning) och metadata‑lista
    chunk_vectors = np.array([item["embedding"] for item in embeddings])
    records = [{"text": item["text"], "source": item["source"]} for item in embeddings]

Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 900038843

## Semantisk sökning funktionerna

In [164]:
def cosine_similarity(vector, matrix):
    # Normalisera vektorer
    vector_norm = vector / np.linalg.norm(vector)
    matrix_norm = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    # Beräkna similarity för alla på en gång
    return np.dot(matrix_norm, vector_norm.T).flatten()

In [165]:
def semantic_search(query, top_k=5):
    query_response = create_embeddings(query)
    query_vector = np.array(query_response.embeddings[0].values).reshape(1,-1)

    scores = cosine_similarity(query_vector, chunk_vectors)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "score": float(scores[idx]),
            "text": records[idx]["text"],
            "source": records[idx]["source"]
        })

    return results

## Kontroll av embeddingsen

In [166]:
def similarity(text_a, text_b):
    emb_a = np.array(create_embeddings(text_a).embeddings[0].values)
    emb_b = np.array(create_embeddings(text_b).embeddings[0].values)
    return cosine_similarity(emb_a, emb_b)


In [167]:
import numpy as np

def similarity(text_a, text_b):
    emb_a = np.array(create_embeddings(text_a).embeddings[0].values)
    emb_b = np.array(create_embeddings(text_b).embeddings[0].values)

    return np.dot(emb_a, emb_b) / (np.linalg.norm(emb_a) * np.linalg.norm(emb_b))

In [170]:
pairs = [
    ("Maskinen fylls inte med vatten.", "Apparaten fylls inte med vatten korrekt."),
    ("Rengör maskinen regelbundet.", "Vänta tills nätspänningen är stabil."),
    ("Avbryt ett pågående program.", "Stoppa maskinen medan den kör."),
]

for a, b in pairs:
    print(f"{a}\n{b}\nLikhet: {similarity(a,b):.2f}\n")


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 10.842353809s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerUserPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-e

In [ ]:
pairs = [
    ("Rengör maskinen regelbundet.", "Maskinen har en maximal kapacitet på 8 kg."),
    ("Avbryt ett pågående program.", "Barn ska inte leka med apparaten."),
    ("Maskinen fylls inte med vatten.", "Apparaten fylls inte med vatten korrekt."),
    ("Hunden är ett djur med fyra ben, en svans och den gillar att äta ben.", "Rengör maskinen regelbundet."),
    ("Maskinen fylls inte med vatten.", "Använd flytande tvättmedel för ullprogram.")
]

for a, b in pairs:
    print(f"{a}\n{b}\nLikhet: {similarity(a,b):.2f}\n")

Rengör maskinen regelbundet.
Maskinen har en maximal kapacitet på 8 kg.
Likhet: 0.80

Avbryt ett pågående program.
Barn ska inte leka med apparaten.
Likhet: 0.78

Maskinen fylls inte med vatten.
Apparaten fylls inte med vatten korrekt.
Likhet: 0.89

Hunden är ett djur med fyra ben, en svans och den gillar att äta ben.
Rengör maskinen regelbundet.
Likhet: 0.69

Maskinen fylls inte med vatten.
Använd flytande tvättmedel för ullprogram.
Likhet: 0.77



In [ ]:
def retrieval_evaluation(query):
    results = semantic_search(query)

    for r in results:
        print(f"Score: {r['score']:.3f}")
        print(f"Source: {r['source']}")
        print(r["text"])
        print("-" * 80)

In [ ]:
query = "Varför fylls inte maskinen med vatten?"

retrieval_evaluation(query)

Score: 0.827
Source: 9000388437_D.pdf
• Strömförsörjning till tvättmaskinen saknas: - Kontrollera om kontakten är isatt. - Kontrollera säkringarna. • Vattenkranen är stängd. • Tillflödesslangen är knäckt. • Locket är inte riktigt stängt.
--------------------------------------------------------------------------------
Score: 0.822
Source: 9000388437_D.pdf
23 24 Vad göra om... Problem Programmet startar inte Starka vibrationer vid centrifugering Tvätten blev inte alls eller inte tillräckligt centrifugerad Vattenläckage från/under maskinen Maskinen pumpas inte ur Locket kan inte öppnas Trumlocken öppnas för långsamt (beroende på modell) Möjlig orsak/åtgärd • Programmet valdes inte fullständigt: - Knappen ”Start/Paus” är inte intryckt. - Programvredet står i stopposition. • Strömförsörjning till tvättmaskinen saknas: - Kontrollera om kontakten är isatt. - Kontrollera säkringarna.
--------------------------------------------------------------------------------
Score: 0.816
Source: 900038843

In [ ]:
query = "Hur rengör jag maskinen?"

retrieval_evaluation(query)

Score: 0.848
Source: 9000388437_D.pdf
20 3 - Rengöring och skötsel Hölje.............................................................................................................................. 21 Manöverpanel, sockel, och övriga inre delar.......................................................... 21 Tvättmedelsbehållare ................................................................................................ 21 Rengöring av filtret .................................................................................................... 22 4 - Blinkande display......................................................................................................
--------------------------------------------------------------------------------
Score: 0.838
Source: 9000388437_D.pdf
• Vi rekommenderar att du regelbundet kör maskinen på vittvättprogrammet 90 °C. Filter i vatteninloppet Vi rekommenderar att filter i vatteninloppet rengörs regelbundet. Tvättmedelsfack Rengör tvättmedels

In [ ]:
query = "Maskinen fylls inte med vatten"

retrieval_evaluation(query)

Score: 0.839
Source: 9000388437_D.pdf
Använd aldrig lösningsmedel till rengöringen. /L60445Frostrisk! Vid frostrisk måste slangen för vattentillförsel kopplas loss och allt vatten i maskinen tömmas ut. På så vis tappas det överskottsvatten ut som kan finnas kvar i behållaren. Hölje Använd endast vatten och tvål.
--------------------------------------------------------------------------------
Score: 0.838
Source: 9000388437_D.pdf
• Strömförsörjning till tvättmaskinen saknas: - Kontrollera om kontakten är isatt. - Kontrollera säkringarna. • Vattenkranen är stängd. • Tillflödesslangen är knäckt. • Locket är inte riktigt stängt.
--------------------------------------------------------------------------------
Score: 0.832
Source: 9000388437_D.pdf
• Filtret på tömningspumpen är igentäppt: - Rengör detta (se kapitlet ”RENGÖRING AV FILTER”) • Avflödesslangen är knäckt eller klämd. • Programmet är inte färdigt än. Locket är låst under hela programförloppet. • Tvättmaskinen har inte använts på l

In [ ]:
query = "Varför startar inte maskinen?"

retrieval_evaluation(query)

Score: 0.838
Source: 9000388437_D.pdf
23 24 Vad göra om... Problem Programmet startar inte Starka vibrationer vid centrifugering Tvätten blev inte alls eller inte tillräckligt centrifugerad Vattenläckage från/under maskinen Maskinen pumpas inte ur Locket kan inte öppnas Trumlocken öppnas för långsamt (beroende på modell) Möjlig orsak/åtgärd • Programmet valdes inte fullständigt: - Knappen ”Start/Paus” är inte intryckt. - Programvredet står i stopposition. • Strömförsörjning till tvättmaskinen saknas: - Kontrollera om kontakten är isatt. - Kontrollera säkringarna.
--------------------------------------------------------------------------------
Score: 0.818
Source: 9000388437_D.pdf
• Strömförsörjning till tvättmaskinen saknas: - Kontrollera om kontakten är isatt. - Kontrollera säkringarna. • Vattenkranen är stängd. • Tillflödesslangen är knäckt. • Locket är inte riktigt stängt.
--------------------------------------------------------------------------------
Score: 0.812
Source: 900038843

In [ ]:
query = "Min tvättmaskin tar inget vatten vad gör jag?"

retrieval_evaluation(query)

Score: 0.876
Source: 9000388437_D.pdf
• Strömförsörjning till tvättmaskinen saknas: - Kontrollera om kontakten är isatt. - Kontrollera säkringarna. • Vattenkranen är stängd. • Tillflödesslangen är knäckt. • Locket är inte riktigt stängt.
--------------------------------------------------------------------------------
Score: 0.863
Source: 9000388437_D.pdf
23 24 Vad göra om... Problem Programmet startar inte Starka vibrationer vid centrifugering Tvätten blev inte alls eller inte tillräckligt centrifugerad Vattenläckage från/under maskinen Maskinen pumpas inte ur Locket kan inte öppnas Trumlocken öppnas för långsamt (beroende på modell) Möjlig orsak/åtgärd • Programmet valdes inte fullständigt: - Knappen ”Start/Paus” är inte intryckt. - Programvredet står i stopposition. • Strömförsörjning till tvättmaskinen saknas: - Kontrollera om kontakten är isatt. - Kontrollera säkringarna.
--------------------------------------------------------------------------------
Score: 0.856
Source: 900038843

Efter första utvärderingen (n=10 och overlap=2) så ser jag att embeddingen verkar fungera bra men chunksen är för stora då det kommer med ganska mycket orelevant information också. 
Så jag väljer att ändra till n=5 men behåller overlap=2.

Efter andra utvärderingen (n=5 och overlap=2) så blir retrivaln mycket sämre. Frågorna om vattnet tappade hittar mest orelevanta saker. Detta kan vara för att problemet och lösningen hamnar i olika chunks. Så mitt nästa test blir att öka overlap till 3 för att se vad det ger för resultat. 